In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pkgutil, importlib
import sys
import json, yaml, os
from silver_transformations import *

In [0]:
from utils import *
logger = get_logger("dataloadinitiate")

In [0]:
schemas_map = load_all_schemas()
print("All schemas loaded:", schemas_map.keys())

In [0]:
dbutils.widgets.text("fileList", "")
dbutils.widgets.text("taxYear", "")
dbutils.widgets.text("clientId", "")
dbutils.widgets.text("env","")
dbutils.widgets.text("bronze_parquet_path", "")
dbutils.widgets.text("maps","")
# fileList = dbutils.widgets.get("fileList")
taxYear = dbutils.widgets.get("taxYear")
clientId = dbutils.widgets.get("clientId")
env = dbutils.widgets.get("env")
bronze_parquet_path = dbutils.widgets.get("bronze_parquet_path")
maps=json.loads(dbutils.widgets.get("maps"))
logger.info(f"taxYear: {taxYear}, clientId: {clientId}, env: {env}, maps: {maps}, bronze_parquet_path: {bronze_parquet_path}")

In [0]:
env_config=load_config(f"{root_dir}/config/etl_main.yaml",env)
pipeline_config=load_config(f"{root_dir}/config/pipeline.yaml",env)

In [0]:
# Dictionary to hold DataFrames
dfs = {}

for map_name in maps:
    # Build parquet path from bronze folder
    parquet_path=f"{bronze_parquet_path}/{map_name}_data"
    print(parquet_path)
    df = spark.read.parquet(parquet_path)
    # get the list of transformations
    transformations = pipeline_config[map_name]
    # apply each transformation
    for transformation in transformations:
        # cconvert to function
        func=getattr(add_audit_columns,transformation)
        # apply function
        df = df.transform(func,f"{map_name}_data")
    # Store in dictionary
    dfs[map_name] = df
    #print(f"df_{map_name} is created and stored in dfs['{map_name}']")
    logger.info(f"df_{map_name} is created and stored in dfs['{map_name}']")


In [0]:
count={}
for name,df in dfs.items():
    df.write.format("delta").options(mergeSchema=True).mode('overwrite').saveAsTable(f"cp_database.{clientId}_{name}")
    count[name]=df.count()
    print(f"{df} written to delta table cp_database.{clientId}_{name}")
    logger.info(f"{df} written to delta table cp_database.{clientId}_{name}")

In [0]:
logger.info(f"exiting the dataloadinitiate notebook")
dbutils.notebook.exit(json.dumps({
    "status": "OK",
    "message": "All files processed successfully",
    "counts": count
}))